# NDR — Notebook 5: Neural mask Fine-Tuning (NFT) — bám SOTA detector

**Deliverable:** model sửa weight (qua mask), gỡ backdoor, GIỮ detect streak. Submit output TRỰC TIẾP. Không boost, không rescoring.

## Điểm neo LB thật
prune-only 5% → **LB maCADD = 261.2** (empty=284, poisoned=379, top≈145). Local val_macadd 1.093 đi CÙNG chiều LB → local là proxy tin cậy.

## Vì sao NFT (khác NB2/3/4)
3 lần fine-tune WEIGHT đều collapse. arXiv 2407.10052 (NFT) chỉ đúng nguyên nhân: *direct weight fine-tuning under limited validation data → poor clean accuracy, primarily due to OVERFITTING*. Giải pháp: **fine-tune MASK per-channel thay vì weight** — không gestrichen được weight gốc nên không overfit; mask chỉ TẮT dần channel bắn poison. + **MixUp** (thay adversarial trigger-synthesis) + **mask regularizer** neo mask≈1 chống drift. NFT test cả trên object detection/PASCAL VOC, chạy được cả khi rất ít clean data — đúng chế độ 20 ảnh.

## Cơ chế
1. Bọc mỗi Conv2d trong cls_subnet + box_subnet bằng MaskedConv2d: weight ĐÓNG BĂNG, chỉ học mask m∈[0,1] mỗi out-channel (init 1).
2. Train trên retain-DÀY (streak thật có nhãn + poison dán không nhãn) với MixUp: loss detection kéo mask channel bắn-poison về 0, giữ channel detect streak ≈1.
3. Mask regularizer λ·mean((m-1)²) chống drift. Backbone freeze.

**Quy tắc nộp:** chỉ nộp khi local val_macadd < 1.093 VÀ retention ≥ 0.6.

## 1. Setup

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

In [ ]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

## 2. Metric core — maCADD (unit-tested)

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = float(cs[i])-float(ps[j])
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

## 3. Load data + poisoned model

In [ ]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

## 4. Validation set (poison + synth streak thật)

In [ ]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

## 5. Clean reference = CHỈ streak thật

In [ ]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

## 6. Thước đo genuine + TTA helpers

In [ ]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

In [ ]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. (Tùy chọn) Activation-guided pruning — dùng làm warm-start nhẹ

In [ ]:
def collect_and_prune(model, prune_frac):
    model.eval()
    layers = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    stored = {i: [] for i in range(len(layers))}; hooks=[]
    for i,ly in enumerate(layers):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts = [{'file_name': str(Path(UNLEARN_DIR)/f'{s}.png'), 'image_id': int(s), 'height': IMG_H, 'width': IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im = load_image(d['file_name']); model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    conv_layers = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    total = 0
    for li,(mi,layer) in enumerate(conv_layers):
        acts = stored[li]; C = acts[0].shape[1]; fg = torch.zeros(C); bg = torch.zeros(C); nf=nb_=0
        rng = np.random.default_rng(SEED)
        for act,d in zip(acts, dicts):
            a = act[0]; _,aH,aW = a.shape; sy,sx = aH/d['height'], aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'], []):
                x1,y1 = max(0,int(x*sx)),max(0,int(y*sy)); x2,y2 = min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw = max(1,int(16*sy)),max(1,int(16*sx)); ry,rx = rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg += a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score = (fg-bg).numpy(); n_prune = max(1,int(len(score)*prune_frac)); bad = np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad] = 0.0
            if layer.bias is not None: layer.bias.data[bad] = 0.0
        total += n_prune
    return model, total

## 8. Baseline poisoned + neo LB

In [ ]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
LB_ANCHOR_MACADD = 1.093   # local val_macadd của p05 (=LB 261.2)
print('POISONED baseline:', {k: round(v,4) for k,v in base_stats.items()})
print('Local neo (p05 = LB261.2): val_macadd =', LB_ANCHOR_MACADD)
print('-> Chỉ nộp NB4 nếu val_macadd < %.3f VÀ retention >= 0.6' % LB_ANCHOR_MACADD)

## 9. Retain set DÀY (in-distribution)

In [ ]:
IMG_STORE = {}
def build_dense_retain(n_imgs, seed=SEED, paste_poison=True):
    rng = np.random.default_rng(seed); records = []; npos = 0; nimg_pois = 0
    for k in range(n_imgs):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); anns = []
        for _ in range(rng.integers(1,4)):
            g2, box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2)
            x1,y1,x2,y2 = box
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],
                             'bbox_mode': BoxMode.XYXY_ABS, 'category_id': 0}); npos += 1
        if paste_poison and rng.random() < 0.7:
            for _ in range(rng.integers(1,3)):
                patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
                spot = rand_free_spot(stem, ph, pw, rng)
                if spot: paste_patch(img, patch, spot[0], spot[1])   # KHÔNG add ann -> forget
            nimg_pois += 1
        iid = 10_000_000 + k; IMG_STORE[iid] = img.astype(np.float32)
        records.append({'image_id': iid, 'height': IMG_H, 'width': IMG_W, 'annotations': anns})
    return records, npos, nimg_pois
_probe, _np, _npi = build_dense_retain(60, seed=1)
print(f'probe: {len(_probe)} ảnh | {_np} pseudo-GT streak ({_np/len(_probe):.2f}/ảnh) | {_npi} ảnh có poison dán')
print('-> so NB2: 0.25/ảnh (collapse). Ở đây dày hơn nhiều.')
IMG_STORE.clear()

## 10. MaskedConv2d — học mask per-channel, ĐÓNG BĂNG weight
Bọc Conv2d: y = conv(x, W_frozen) * sigmoid(a)[None,:,None,None]. Chỉ 'a' (logit mask) học được, init lớn -> mask≈1 lúc đầu (giữ nguyên model gốc).

In [ ]:
class MaskedConv2d(nn.Module):
    def __init__(self, conv):
        super().__init__()
        self.conv = conv
        for p in self.conv.parameters(): p.requires_grad = False
        self.mask_logit = nn.Parameter(torch.full((conv.out_channels,), 4.0))  # sigmoid(4)~0.982
    def forward(self, x):
        return self.conv(x) * torch.sigmoid(self.mask_logit).view(1, -1, 1, 1)

def wrap_masks(model, subnets=('cls_subnet','bbox_subnet')):
    n_masked = 0
    for sn in subnets:
        seq = getattr(model.head, sn, None)
        if seq is None: continue
        for idx, layer in enumerate(seq):
            if isinstance(layer, nn.Conv2d):
                seq[idx] = MaskedConv2d(layer); n_masked += 1
    return model, n_masked

def mask_params(model): return [p for n,p in model.named_parameters() if n.endswith('mask_logit')]

def mask_stats(model):
    allm = torch.cat([torch.sigmoid(p).detach() for p in mask_params(model)])
    return dict(n=allm.numel(), mean=round(float(allm.mean()),4),
                lt0_5=int((allm<0.5).sum()), lt0_2=int((allm<0.2).sum()))

def bake_masked_state(masked_model):
    '''Nướng mask vào weight -> state_dict RetinaNet CHUẨN (không còn MaskedConv2d),
    để build_predictor load lại bình thường.'''
    std = build_model(build_cfg(POISONED_WEIGHTS))
    msrc = dict(masked_model.named_modules())
    with torch.no_grad():
        for name, mod in std.named_modules():
            if not isinstance(mod, nn.Conv2d): continue
            src = msrc.get(name)
            if isinstance(src, MaskedConv2d):
                m = torch.sigmoid(src.mask_logit).detach().cpu()
                mod.weight.copy_(src.conv.weight.detach().cpu() * m.view(-1,1,1,1))
                if mod.bias is not None and src.conv.bias is not None:
                    mod.bias.copy_(src.conv.bias.detach().cpu() * m)
            elif isinstance(src, nn.Conv2d):
                mod.weight.copy_(src.weight.detach().cpu())
                if mod.bias is not None and src.bias is not None:
                    mod.bias.copy_(src.bias.detach().cpu())
    # copy phần còn lại (backbone/fpn/norm...) từ model masked theo tên khớp shape
    tgt = std.state_dict(); msd = masked_model.state_dict()
    for k in tgt:
        if k in msd and tgt[k].shape == msd[k].shape:
            tgt[k] = msd[k].detach().cpu()
    # ghi đè lại conv đã nướng (ưu tiên)
    for name, mod in std.named_modules():
        if isinstance(mod, nn.Conv2d):
            tgt[name+'.weight'] = mod.weight.detach().cpu()
            if mod.bias is not None: tgt[name+'.bias'] = mod.bias.detach().cpu()
    del std; return tgt
print('MaskedConv2d + bake defined.')

## 11. Train NFT: chỉ học mask trên retain-DÀY + MixUp + mask-reg
MixUp trộn 2 ảnh retain (arXiv 2407.10052 dùng thay adversarial trigger-synthesis). Chỉ optimizer trên mask_logit. Backbone+weight đóng băng.

In [ ]:
from detectron2.utils.events import EventStorage
def collate_simple(batch): return batch
def build_loader(records, batch=BATCH_SIZE):
    register_dense(records)
    cfg = build_cfg(POISONED_WEIGHTS)
    cfg.DATASETS.TRAIN=(RETAIN_DS,); cfg.DATALOADER.NUM_WORKERS=2; cfg.SOLVER.IMS_PER_BATCH=batch
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
    return build_detection_train_loader(cfg, mapper=DenseMapper(cfg,is_train=True,augmentations=[]),
                                        dataset=DatasetCatalog.get(RETAIN_DS))

def train_nft(n_retain, lr, iters, mask_reg, mixup_alpha, prune_warm, seed, tag):
    out = WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = build_model(build_cfg(POISONED_WEIGHTS, output_dir=out))
    DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    if prune_warm > 0: model, _ = collect_and_prune(model, prune_warm)
    model, n_masked = wrap_masks(model); model.to(DEVICE)
    for n,p in model.named_parameters(): p.requires_grad = n.endswith('mask_logit')
    opt = torch.optim.Adam(mask_params(model), lr=lr)
    IMG_STORE.clear(); records,_,_ = build_dense_retain(n_retain, seed=seed)
    loader = build_loader(records); it = iter(loader)
    rng = np.random.default_rng(seed); model.train()
    with EventStorage(0):
        for step in range(iters):
            try: data = next(it)
            except StopIteration: it = iter(loader); data = next(it)
            # MixUp mức ẢNH (chỉ trộn pixel, giữ union nhãn để không mất GT streak)
            if mixup_alpha > 0 and len(data) >= 2:
                lam = float(rng.beta(mixup_alpha, mixup_alpha)); lam = max(lam, 1-lam)
                j = int(rng.integers(1, len(data)))
                for a in range(len(data)):
                    b = (a + j) % len(data)
                    ia, ib = data[a]['image'].float(), data[b]['image'].float()
                    if ia.shape == ib.shape: data[a]['image'] = lam*ia + (1-lam)*ib
            ld = model(data); det_loss = sum(v for v in ld.values())
            reg = torch.stack([((torch.sigmoid(p)-1.0)**2).mean() for p in mask_params(model)]).mean()
            loss = det_loss + mask_reg*reg
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for p in mask_params(model): p.clamp_(-8, 8)
    st = mask_stats(model)
    std_sd = bake_masked_state(model)
    final = str(out/'model_final.pth'); torch.save({'model': std_sd}, final)
    del model; torch.cuda.empty_cache()
    return final, st

## 12. Sweep NFT — chọn theo val_macadd (giữ streak + xóa poison)

In [ ]:
SWEEP = [
    dict(n_retain=250, lr=0.05, iters=300, mask_reg=0.5, mixup_alpha=0.0, prune_warm=0.0, seed=42, tag='m0'),
    dict(n_retain=250, lr=0.05, iters=400, mask_reg=1.0, mixup_alpha=0.2, prune_warm=0.0, seed=42, tag='m1'),
    dict(n_retain=300, lr=0.10, iters=400, mask_reg=0.5, mixup_alpha=0.2, prune_warm=0.05, seed=7, tag='m2'),
    dict(n_retain=300, lr=0.10, iters=500, mask_reg=2.0, mixup_alpha=0.4, prune_warm=0.0, seed=13, tag='m3'),
]
sweep_results = []
for spec in SWEEP:
    w, mst = train_nft(**spec)
    pr = build_predictor(w, CAND_THRESH); stats = eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    sweep_results.append({**spec, 'weights': w, **stats, 'mask_mean': mst['mean'], 'mask_off': mst['lt0_2']})
    print(f"{spec['tag']}: firing={stats['poison_firing_rate']:.3f} "
          f"retention={stats['streak_retention']:.3f} val_macadd={stats['val_macadd']:.4f} "
          f"| mask_mean={mst['mean']} off(<0.2)={mst['lt0_2']}/{mst['n']}")
sweep_df = pd.DataFrame(sweep_results).sort_values('val_macadd')
print('\n', sweep_df[['tag','n_retain','lr','iters','mask_reg','mixup_alpha','poison_firing_rate','streak_retention','val_macadd','mask_off']].to_string(index=False))
print(f"\nPOISONED baseline: firing={base_stats['poison_firing_rate']:.3f} retention={base_stats['streak_retention']:.3f} val_macadd={base_stats['val_macadd']:.4f}")
print(f"neo p05 (LB261.2): val_macadd={LB_ANCHOR_MACADD}")

## 13. Chọn cấu hình thắng

In [ ]:
cands = [r for r in sweep_results if r['streak_retention'] >= 0.6 and r['val_macadd'] < LB_ANCHOR_MACADD]
pool = cands if cands else sweep_results
best = min(pool, key=lambda r: r['val_macadd'])
FINAL_WEIGHTS = best['weights']
FINAL_STATS = {k:best[k] for k in ('poison_firing_rate','streak_retention','val_macadd')}
print('CHỌN:', best['tag'], {k: round(float(v),4) for k,v in FINAL_STATS.items()})
if not cands: print('(chưa bản nào beat neo + retention>=0.6 -> tham khảo, xem quy tắc nộp cuối)')

## 14. TTA D4+WBF (chỉ nhận nếu cải thiện validation)

In [ ]:
final_pr = build_predictor(FINAL_WEIGHTS, CAND_THRESH)
tta_stats = eval_model_on_val(final_pr, VAL, tta=True)
print('no-TTA :', {k: round(float(v),4) for k,v in FINAL_STATS.items()})
print('TTA    :', {k: round(v,4) for k,v in tta_stats.items()})
USE_TTA = tta_stats['val_macadd'] <= FINAL_STATS['val_macadd']
print('-> dùng TTA:', USE_TTA)

## 15. Submission = output trực tiếp model NFT (mask đã nướng)

In [ ]:
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id': str})
def fmt(b, s):
    parts = []
    for (x1,y1,x2,y2), sc in zip(b, s):
        x1,y1 = float(np.clip(x1,0,IMG_W)), float(np.clip(y1,0,IMG_H))
        x2,y2 = float(np.clip(x2,0,IMG_W)), float(np.clip(y2,0,IMG_H))
        w,h = x2-x1, y2-y1
        if w>0 and h>0 and sc>0: parts += [f'{float(sc):.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{w:.2f}', f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred = {}
for p in tqdm(test_files, desc='infer test'):
    img = load_image(p)
    b,s = predict_tta(final_pr, img) if USE_TTA else predict_image(final_pr, img)
    keep = s > SUBMIT_THRESH; pred[p.stem] = (b[keep], s[keep])
sub = sample.copy()
sub['prediction_string'] = sub['image_id'].map(lambda i: fmt(*pred.get(str(i), E)))
sub['prediction_string'] = sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string'] = ' '
sub.to_csv(SUBMISSION_PATH, index=False)
total = sum(len(s) for _,s in pred.values()); npred = int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | rows={len(sub)} | dets={total} ({total/len(test_files):.3f}/img) | imgs w/ preds: {npred}')
print('so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | p05(LB261)=0.371/img')

## 16. Tổng kết + quy tắc nộp

In [ ]:
print('='*60)
print('POISONED   :', {k: round(v,4) for k,v in base_stats.items()})
print('DE-POISONED:', {k: round(float(v),4) for k,v in FINAL_STATS.items()})
print('neo p05    : val_macadd=%.3f (LB=261.2)' % LB_ANCHOR_MACADD)
better = FINAL_STATS['val_macadd'] < LB_ANCHOR_MACADD
ret_ok = FINAL_STATS['streak_retention'] >= 0.6
dets_per_img = total/len(test_files)
ok = better and ret_ok and (0.5 <= dets_per_img <= 3.5)
print('\nĐẠT TIÊU CHÍ NỘP:', ok)
if ok:
    print(' -> NÊN nộp: NFT beat neo p05 trên local, retention giữ, dets/img hợp lý.')
else:
    print(' -> CHƯA nộp (không tốt hơn p05). Giữ p05 là bản tốt nhất.')
    print('    Tune: tăng lr/iters nếu mask không tắt channel; giảm mask_reg nếu firing còn cao;')
    print('          tăng mask_reg/giảm iters nếu retention rớt; thử prune_warm=0.05.')